In [1]:
def main(datasources, start_date, end_date):
    """
    极简测试因子：过去5日平均收益率（已修复排序问题）
    """
    import numpy as np
    import pandas as pd
    try:
        import dai
    except Exception:
        dai = None

    start_ts = pd.Timestamp(start_date).normalize()
    end_ts = pd.Timestamp(end_date).normalize()
    query_start = (start_ts - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
    query_end = (end_ts + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    # ---------- 辅助函数 ----------
    def _resolve(names, default):
        if isinstance(datasources, dict):
            for name in names:
                value = datasources.get(name)
                if value is not None:
                    return value
        return default

    def _query(names, default_table, fields, left, right):
        source = _resolve(names, default_table)
        if isinstance(source, pd.DataFrame):
            frame = source.copy()
        elif hasattr(source, "query") and not isinstance(source, str):
            try:
                frame = source.query(
                    "SELECT " + ", ".join(fields),
                    filters={"date": [left, right]}
                ).df()
            except Exception:
                frame = pd.DataFrame()
        elif dai is not None:
            try:
                frame = dai.query(
                    "SELECT " + ", ".join(fields) + " FROM " + str(source),
                    filters={"date": [left, right]},
                    compression=True,
                ).df()
            except Exception:
                frame = pd.DataFrame()
        else:
            frame = pd.DataFrame()
        
        if frame.empty:
            return frame
        
        frame = frame.copy()
        if "date" in frame.columns:
            frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
            frame = frame[(frame["date"] >= pd.Timestamp(left)) & 
                          (frame["date"] < pd.Timestamp(right))]
        if "instrument" in frame.columns:
            frame["instrument"] = frame["instrument"].astype(str)
        return frame

    # ---------- 1. 获取股票池 ----------
    instruments = _query(
        ["instruments", "instrument", "bigalpha_2026_instruments"],
        "bigalpha_2026_instruments",
        ["date", "instrument"],
        query_start,
        query_end,
    )
    if instruments.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    
    instruments["date"] = pd.to_datetime(instruments["date"]).dt.normalize()
    panel = instruments.dropna(subset=["date", "instrument"]).drop_duplicates(
        ["date", "instrument"]
    ).copy()

    # ---------- 2. 获取行情数据 ----------
    bar = _query(
        ["bar1m", "stock_bar1m", "bigalpha_2026_stock_bar1m"],
        "bigalpha_2026_stock_bar1m",
        ["date", "instrument", "close"],
        query_start,
        query_end,
    )
    
    if bar.empty:
        panel["factor"] = 0.5
    else:
        # ---- 关键修复1：在分组聚合前显式排序 ----
        bar = bar.sort_values(["instrument", "date"]).copy()
        bar["date"] = pd.to_datetime(bar["date"]).dt.normalize()
        
        # 计算日收盘价（取每天最后一笔），此处先排序再分组
        daily_close = bar.sort_values(["instrument", "date"]).groupby(
            ["date", "instrument"], sort=False
        )["close"].last().reset_index()
        daily_close["close"] = pd.to_numeric(daily_close["close"], errors="coerce")
        
        panel = panel.merge(daily_close, on=["date", "instrument"], how="left")
        
        # ---------- 3. 计算过去5日平均收益率 ----------
        # ---- 关键修复2：确保 panel 按 instrument 和 date 排序 ----
        panel = panel.sort_values(["instrument", "date"])
        panel["return"] = panel.groupby("instrument")["close"].pct_change()
        
        # 滚动窗口计算均值
        panel["factor"] = panel.groupby("instrument")["return"].transform(
            lambda x: x.rolling(5, min_periods=2).mean()
        )
        panel["factor"] = panel["factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # ---------- 4. 输出最终结果 ----------
    out = panel.loc[
        (panel["date"] >= start_ts) & (panel["date"] <= end_ts),
        ["date", "instrument", "factor"]
    ].copy()
    
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    out["instrument"] = out["instrument"].astype(str)
    out["factor"] = pd.to_numeric(out["factor"], errors="coerce").fillna(0.5)
    
    return out.dropna(subset=["date", "instrument"]).drop_duplicates(
        ["date", "instrument"], keep="last"
    ).sort_values(["date", "instrument"]).reset_index(drop=True)